<a href="https://colab.research.google.com/github/itsdakshjain/Smartphone-Data-Cleaning-and-EDA/blob/main/DataCleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📱 Smartphone Data Cleaning: From Raw Scrape to Analysis
**The Goal:** I’m taking 1,000+ rows of raw smartphone data scraped from Smartprix and cleaning it up so I can actually use it for analysis.

### 🛠️ What I'm solving:
Scraped data is never perfect. To get this dataset ready, I have to handle three main problems:
* **Nested Info:** Things like RAM, Battery, and Camera are all stuck in single strings. I need to pull them out into separate columns.
* **Shifted Rows:** The scraper occasionally put data in the wrong columns (like battery info showing up in the 'Display' column).
* **Junk Data:** I found things like iPods and feature phones that don't belong in a smartphone price analysis.

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
#pd.set_option('display.max_columns', None)

In [2]:
url = 'https://raw.githubusercontent.com/itsdakshjain/Smartphone-Data-Cleaning-and-EDA/refs/heads/main/smartphone.csv'
df = pd.read_csv(url)

df.head()

,model,price,rating,sim,processor,ram,battery,display,camera,card,os
0,OnePlus 11 5G,"₹54,999",89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen2, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",5000 mAh Battery with 100W Fast Charging,"6.7 inches, 1440 x 3216 px, 120 Hz Display wit...",50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,Memory Card Not Supported,Android v13
1,OnePlus Nord CE 2 Lite 5G,"₹19,989",81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 33W Fast Charging,"6.59 inches, 1080 x 2412 px, 120 Hz Display wi...",64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
2,Samsung Galaxy A14 5G,"₹16,499",75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Exynos 1330, Octa Core, 2.4 GHz Processor","4 GB RAM, 64 GB inbuilt",5000 mAh Battery with 15W Fast Charging,"6.6 inches, 1080 x 2408 px, 90 Hz Display with...",50 MP + 2 MP + 2 MP Triple Rear & 13 MP Front ...,"Memory Card Supported, upto 1 TB",Android v13
3,Motorola Moto G62 5G,"₹14,999",81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.55 inches, 1080 x 2400 px, 120 Hz Display wi...",50 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
4,Realme 10 Pro Plus,"₹24,999",82.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Dimensity 1080, Octa Core, 2.6 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 67W Fast Charging,"6.7 inches, 1080 x 2412 px, 120 Hz Display wit...",108 MP + 8 MP + 2 MP Triple Rear & 16 MP Front...,Memory Card Not Supported,Android v13


##  What's in this dataset?
The raw data has about 1020 phones with 11 columns like Model, Price, and Specs. But most of the data is "hidden" inside long text strings.

To do any real analysis, I need to break those 11 columns into **23 specific features**. Here is the plan:

* **Price & Ratings:** I'll clean the `price` (INR) and `rating` columns so they are ready for math.
* **Performance:** Extract `processor_name`, `processor_core`, and `processor_speed` (GHz).
* **Memory:** Split the "Memory" string into `ram_capacity` and `storage_capacity`.
* **Connectivity:** Create simple True/False columns for `has_5g`, `has_nfc`, and `has_ir_blaster`, plus check if `card_supported` is available.
* **Battery:** Pull out the `battery_capacity` (mAh) and the `fast_charging` wattage (W).
* **Display:** I need to find the `display_size`, `resolution` (width/height), `refresh_rate`, and `resolution_type` (like HD+ or FHD+).
* **Camera:** This is the messiest part. I'll split it into `no_of_rear_camera`, `max_rear_camera_MP`, and do the same for the front cameras.
* **Categorical Info:** Keep the `brand_name`, `model`, `sim_type`, and `os` for easy filtering.

#  **Data Assessing**

### **Quality Issues**

* **model**: Some brands use inconsistent styles (e.g., **Oppo** & **OPPO**) `consistency`.

* **price**: Contains unrealistic values (Namotel at ₹99) and symbols (₹, ,) that make it a string `validity`.

* **rating**: Missing values across multiple rows `completeness`.

* **sim**: Row **(756)** is an iPod, causing incorrect SIM data `validity`.

* **processor**: Incorrect values in rows: -> (378, 475, 534, 553, 575, 584, 610, 613, 642, 647, 649, 659, 667, 701, 750, 756, 759, 819, 859, 883, 884, 919, 929, 932, 990, 1002) `validity`.

* **memory**: Incorrect values in rows: -> (378, 441, 485, 534, 553, 584, 610, 642, 647, 649, 659, 667, 701, 750, 759, 819, 859, 884, 919, 927, 929, 932, 990, 1002) `validity`.

* **battery**: Incorrect values in rows: -> (113, 151, 309, 365, 378, 441, 450, 553, 584, 610, 613, 630, 642, 647, 649, 659, 667, 701, 750, 756, 759, 764, 819, 855, 859, 884, 915, 916, 927, 929, 932, 990, 1002) `validity`.

* **display**: Values shifted in rows: -> (378, 441, 450, 553, 584, 610, 613, 630, 642, 647, 649, 659, 667, 701, 750, 759, 764, 819, 859, 884, 915, 916, 927, 929, 932, 990, 1002). Also missing refresh rate info `validity/completeness`.

* **camera**: Uses text (Dual,Triple,Quad) instead of numbers. Issues in rows: -> (100, 113, 151, 157, 161, 238, 273, 308, 309, 323, 324, 365, 367, 378, 394, 441, 450, 484, 506, 534, 553, 571, 572, 575, 584, 610, 613, 615, 630, 642, 647, 649, 659, 667, 684, 687, 705, 711, 723, 728, 750, 756, 759, 764, 792, 819, 846, 854, 855, 858, 883, 884, 896, 915, 916, 927, 929, 932, 945, 956, 990, 995, 1002, 1016) `validity`.

* **OS**: Contains Bluetooth/FM info in rows (324, 378) and inconsistent version names (e.g., Lollipop) `validity/consistency`.

* **General**: Missing values in `camera`, `card`, `os` and incorrect datatypes for `price` and `rating` `completeness/validity`.
---

### **Tidiness Issues**

* **sim**: Need separate columns for 5G, NFC, and IR Blaster.
* **processor**: Need separate columns for name, cores, and speed.
* **ram**: Can be split into **RAM** and **ROM**.
    * *Careful: If RAM value is missing, ROM value might shift into the RAM column.*
* **battery**: Need separate columns for capacity and fast charging wattage.
* **display**: Need separate columns for size, resolution, and refresh rate.
* **camera**: Needs to be split into front and rear camera megapixels.
* **card**: Split into `supported` and `extended_upto`.

In [3]:
# make a copy
df1 = df.copy()

In [4]:
df1.shape

(1020, 11)

In [5]:
df1.info() # checked rating has many null values and camera, card , os too . Price need to be int and rating too ( rating had many null values so pandas made it float)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   model      1020 non-null   object 
 1   price      1020 non-null   object 
 2   rating     879 non-null    float64
 3   sim        1020 non-null   object 
 4   processor  1020 non-null   object 
 5   ram        1020 non-null   object 
 6   battery    1020 non-null   object 
 7   display    1020 non-null   object 
 8   camera     1019 non-null   object 
 9   card       1013 non-null   object 
 10  os         1003 non-null   object 
dtypes: float64(1), object(10)
memory usage: 87.8+ KB


In [6]:
df1.describe() # can't do much due to null values

,rating
count,879.000000
mean,78.258248
std,7.402854
min,60.000000
25%,74.000000
50%,80.000000
75%,84.000000
max,89.000000


In [7]:
df1.duplicated().sum()

np.int64(0)

In [8]:
df1 = df1.reset_index()

In [9]:
df1['index'] = df1['index'] + 2 #our csv file has index from 2 and our mannua assesment porblematics rows follows that order so convert it to be same

In [10]:
df1

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
0,2,OnePlus 11 5G,"₹54,999",89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen2, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",5000 mAh Battery with 100W Fast Charging,"6.7 inches, 1440 x 3216 px, 120 Hz Display wit...",50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,Memory Card Not Supported,Android v13
1,3,OnePlus Nord CE 2 Lite 5G,"₹19,989",81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 33W Fast Charging,"6.59 inches, 1080 x 2412 px, 120 Hz Display wi...",64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
2,4,Samsung Galaxy A14 5G,"₹16,499",75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Exynos 1330, Octa Core, 2.4 GHz Processor","4 GB RAM, 64 GB inbuilt",5000 mAh Battery with 15W Fast Charging,"6.6 inches, 1080 x 2408 px, 90 Hz Display with...",50 MP + 2 MP + 2 MP Triple Rear & 13 MP Front ...,"Memory Card Supported, upto 1 TB",Android v13
3,5,Motorola Moto G62 5G,"₹14,999",81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.55 inches, 1080 x 2400 px, 120 Hz Display wi...",50 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
4,6,Realme 10 Pro Plus,"₹24,999",82.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Dimensity 1080, Octa Core, 2.6 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 67W Fast Charging,"6.7 inches, 1080 x 2412 px, 120 Hz Display wit...",108 MP + 8 MP + 2 MP Triple Rear & 16 MP Front...,Memory Card Not Supported,Android v13
...,...,...,...,...,...,...,...,...,...,...,...,...
1015,1017,Motorola Moto Edge S30 Pro,"₹34,990",83.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","8 GB RAM, 128 GB inbuilt",5000 mAh Battery with 68.2W Fast Charging,"6.67 inches, 1080 x 2460 px, 120 Hz Display wi...",64 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,Android v12,No FM Radio
1016,1018,Honor X8 5G,"₹14,990",75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 480+, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 22.5W Fast Charging,"6.5 inches, 720 x 1600 px Display with Water D...",48 MP + 2 MP + Depth Sensor Triple Rear & 8 MP...,"Memory Card Supported, upto 1 TB",Android v11
1017,1019,POCO X4 GT 5G (8GB RAM + 256GB),"₹28,990",85.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Dimensity 8100, Octa Core, 2.85 GHz Processor","8 GB RAM, 256 GB inbuilt",5080 mAh Battery with 67W Fast Charging,"6.6 inches, 1080 x 2460 px, 144 Hz Display wit...",64 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,Memory Card Not Supported,Android v12
1018,1020,Motorola Moto G91 5G,"₹19,990",80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.8 inches, 1080 x 2400 px Display with Punch ...",108 MP + 8 MP + 2 MP Triple Rear & 32 MP Front...,"Memory Card Supported, upto 1 TB",Android v12


In [11]:
df1['price'] = df1['price'].str.replace('₹','').str.replace(',','').astype('int') # convert the price column

In [12]:
df1

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
0,2,OnePlus 11 5G,54999,89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen2, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",5000 mAh Battery with 100W Fast Charging,"6.7 inches, 1440 x 3216 px, 120 Hz Display wit...",50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,Memory Card Not Supported,Android v13
1,3,OnePlus Nord CE 2 Lite 5G,19989,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 33W Fast Charging,"6.59 inches, 1080 x 2412 px, 120 Hz Display wi...",64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
2,4,Samsung Galaxy A14 5G,16499,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Exynos 1330, Octa Core, 2.4 GHz Processor","4 GB RAM, 64 GB inbuilt",5000 mAh Battery with 15W Fast Charging,"6.6 inches, 1080 x 2408 px, 90 Hz Display with...",50 MP + 2 MP + 2 MP Triple Rear & 13 MP Front ...,"Memory Card Supported, upto 1 TB",Android v13
3,5,Motorola Moto G62 5G,14999,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.55 inches, 1080 x 2400 px, 120 Hz Display wi...",50 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
4,6,Realme 10 Pro Plus,24999,82.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Dimensity 1080, Octa Core, 2.6 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 67W Fast Charging,"6.7 inches, 1080 x 2412 px, 120 Hz Display wit...",108 MP + 8 MP + 2 MP Triple Rear & 16 MP Front...,Memory Card Not Supported,Android v13
...,...,...,...,...,...,...,...,...,...,...,...,...
1015,1017,Motorola Moto Edge S30 Pro,34990,83.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","8 GB RAM, 128 GB inbuilt",5000 mAh Battery with 68.2W Fast Charging,"6.67 inches, 1080 x 2460 px, 120 Hz Display wi...",64 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,Android v12,No FM Radio
1016,1018,Honor X8 5G,14990,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 480+, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 22.5W Fast Charging,"6.5 inches, 720 x 1600 px Display with Water D...",48 MP + 2 MP + Depth Sensor Triple Rear & 8 MP...,"Memory Card Supported, upto 1 TB",Android v11
1017,1019,POCO X4 GT 5G (8GB RAM + 256GB),28990,85.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Dimensity 8100, Octa Core, 2.85 GHz Processor","8 GB RAM, 256 GB inbuilt",5080 mAh Battery with 67W Fast Charging,"6.6 inches, 1080 x 2460 px, 144 Hz Display wit...",64 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,Memory Card Not Supported,Android v12
1018,1020,Motorola Moto G91 5G,19990,80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.8 inches, 1080 x 2400 px Display with Punch ...",108 MP + 8 MP + 2 MP Triple Rear & 32 MP Front...,"Memory Card Supported, upto 1 TB",Android v12


In [13]:
processor_rows = set((642,647,649,659,667,701,750,759,819,859,883,884,919,927,929,932,1002)) # making set of all probalematic rows
ram_rows = set((441,485,534,553,584,610,613,642,647,649,659,667,701,750,759,819,859,884,919,927,929,932,990,1002))
battery_rows = set((113,151,309,365,378,441,450,553,584,610,613,630,642,647,649,659,667,701,750,756,759,764,819,855,859,884,915,916,927,929,932,990,1002))
display_rows = set((378,441,450,553,584,610,613,630,642,647,649,659,667,701,750,759,764,819,859,884,915,916,927,929,932,990,1002))
camera_rows = set((100,113,151,157,161,238,273,308,309,323,324,365,367,378,394,441,450,484,506,534,553,571,572,575,584,610,613,615,630,642,647,649,659,667,684,687,705,711,723,728,750,756,759,764,792,819,846,854,855,858,883,884,896,915,916,927,929,932,945,956,990,995,1002,1016 ))

In [14]:
df1[df1['index'].isin(processor_rows | ram_rows | battery_rows | display_rows | camera_rows)] # union of set showing all rows having atleast 1 issue

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
98,100,Vivo X Fold 5G,106990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","12 GB RAM, 256 GB inbuilt",4600 mAh Battery with 66W Fast Charging,"8.03 inches, 1916 x 2160 px, 120 Hz Display",Foldable Display,50 MP Quad Rear & 16 MP Front Camera,Android v12
111,113,Apple iPhone 12,51999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt","6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14,No FM Radio
149,151,Apple iPhone 12 Mini,40999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt","5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14,No FM Radio
155,157,Nokia 2780 Flip,4990,NaN,"Dual Sim, 3G, 4G, Wi-Fi","Snapdragon QM215, Quad Core, 1.3 GHz Processor","4 GB RAM, 512 MB inbuilt",1450 mAh Battery,"2.7 inches, 240 x 320 px Display",Dual Display,5 MP Rear Camera,"Memory Card Supported, upto 32 GB"
159,161,Oppo Find N2 5G,94990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",4520 mAh Battery with 67W Fast Charging,"7.1 inches, 1792 x 1920 px, 120 Hz Display wit...","Foldable Display, Dual Display",50 MP + 48 MP + 32 MP Triple Rear & 32 MP + 32...,Memory Card Not Supported
...,...,...,...,...,...,...,...,...,...,...,...,...
954,956,Vivo X Fold 5G (12GB RAM + 512GB),118990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","12 GB RAM, 512 GB inbuilt",4600 mAh Battery with 66W Fast Charging,"8.03 inches, 1916 x 2160 px, 120 Hz Display",Foldable Display,50 MP Quad Rear & 16 MP Front Camera,Android v12
988,990,Nokia 5310 Dual Sim,3399,NaN,Dual Sim,"8 MB RAM, 16 MB inbuilt",1200 mAh Battery,"2.4 inches, 240 x 320 px Display",0.3 MP Rear Camera,"Memory Card Supported, upto 32 GB",Bluetooth,Browser
993,995,Huawei Mate X,169000,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Kirin 990, Octa Core, 2.86 GHz Processor","8 GB RAM, 512 GB inbuilt",4500 mAh Battery with 55W Fast Charging,"8 inches, 2200 x 2480 px Display",Foldable Display,48 MP Quad Rear Camera,"Memory Card (Hybrid), upto 256 GB"
1000,1002,XTouch F40 Flip,1999,NaN,Dual Sim,No 3G,No Wifi,"32 MB RAM, 32 MB inbuilt",800 mAh Battery,"1.77 inches, 240 x 320 px Display",Dual Display,1.3 MP Rear Camera


In [15]:
df1[df1['index'].isin(processor_rows & ram_rows & battery_rows & display_rows & camera_rows)]# intersection showing rows having issue in all column (data is shifted due to webscrapping)
# after finding its price mean , we get to know its 2500 and we observe its mostly feature phones below 3400

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
640,642,Nokia 105 Plus,1299,NaN,Dual Sim,"4 MB RAM, 4 MB inbuilt",800 mAh Battery,"1.77 inches, 128 x 160 px Display",No Rear Camera,"Memory Card Supported, upto 32 GB",Bluetooth,NaN
645,647,Nokia 2760 Flip,5490,NaN,"Dual Sim, 3G, 4G, Wi-Fi",1450 mAh Battery,"3.6 inches, 240 x 320 px Display",5 MP Rear & 5 MP Front Camera,"Memory Card Supported, upto 32 GB",Kaios v3.0,Bluetooth,NaN
647,649,Motorola Moto A10,1339,NaN,Dual Sim,"4 MB RAM, 4 MB inbuilt",1750 mAh Battery,"1.8 inches, 160 x 128 px Display",No Rear Camera,"Memory Card Supported, upto 32 GB",NaN,NaN
657,659,Zanco Tiny T1,2799,NaN,Single Sim,"32 MB RAM, 32 MB inbuilt",200 mAh Battery,"0.49 inches, 64 x 32 px Display",No Rear Camera,No FM Radio,Bluetooth,NaN
665,667,itel it2163S,958,NaN,Dual Sim,"4 MB RAM, 4 MB inbuilt",1200 mAh Battery,"1.8 inches, 160 x 128 px Display",No Rear Camera,"Memory Card Supported, upto 32 GB",Bluetooth,NaN
748,750,Nokia 400 4G,3290,NaN,"Dual Sim, 4G, VoLTE, Wi-Fi",2000 mAh Battery,"2.4 inches, 240 x 320 px Display",0.3 MP Rear & 0.3 MP Front Camera,"Memory Card Supported, upto 64 GB",Bluetooth,Browser,NaN
757,759,Karbonn KU3i,995,NaN,Dual Sim,"52 MB RAM, 32 MB inbuilt",1000 mAh Battery,"1.8 inches, 128 x 160 px Display",No Rear Camera,"Memory Card Supported, upto 16 GB",Bluetooth,NaN
817,819,itel Magic X,2239,NaN,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi",No 3G,T117,"48 MB RAM, 128 MB inbuilt",1200 mAh Battery,"2.4 inches, 240 x 320 px Display",1.3 MP Rear Camera,"Memory Card Supported, upto 64 GB"
882,884,Nokia 5710 XpressAudio,4799,NaN,"Dual Sim, 3G, 4G",No Wifi,Unisoc T107,"48 MB RAM, 128 MB inbuilt",1450 mAh Battery,"2.4 inches, 240 x 320 px Display",0.3 MP Rear Camera,"Memory Card Supported, upto 32 GB"
925,927,Nokia 3310 4G,3999,NaN,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi","256 MB RAM, 512 MB inbuilt",1200 mAh Battery,"2.4 inches, 240 x 320 px Display",2 MP Rear Camera,"Memory Card Supported, upto 32 GB",Bluetooth,Browser


In [16]:
df1 = df1[df1['price'] >= 3400] # we are only keeping those above 3400 ,

In [17]:
df1[df1['index'].isin(processor_rows & ram_rows & battery_rows & display_rows & camera_rows)].shape # reduced 13 intersected problem rows to only 3

(3, 12)

In [18]:
df1[df1['index'].isin(processor_rows | ram_rows | battery_rows | display_rows | camera_rows)].shape # reduced 68 intersected problem rows to only 47

(47, 12)

In [19]:
df1 # reduced 1020 total rows to 991

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
0,2,OnePlus 11 5G,54999,89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen2, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",5000 mAh Battery with 100W Fast Charging,"6.7 inches, 1440 x 3216 px, 120 Hz Display wit...",50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,Memory Card Not Supported,Android v13
1,3,OnePlus Nord CE 2 Lite 5G,19989,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 33W Fast Charging,"6.59 inches, 1080 x 2412 px, 120 Hz Display wi...",64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
2,4,Samsung Galaxy A14 5G,16499,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Exynos 1330, Octa Core, 2.4 GHz Processor","4 GB RAM, 64 GB inbuilt",5000 mAh Battery with 15W Fast Charging,"6.6 inches, 1080 x 2408 px, 90 Hz Display with...",50 MP + 2 MP + 2 MP Triple Rear & 13 MP Front ...,"Memory Card Supported, upto 1 TB",Android v13
3,5,Motorola Moto G62 5G,14999,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.55 inches, 1080 x 2400 px, 120 Hz Display wi...",50 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
4,6,Realme 10 Pro Plus,24999,82.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Dimensity 1080, Octa Core, 2.6 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 67W Fast Charging,"6.7 inches, 1080 x 2412 px, 120 Hz Display wit...",108 MP + 8 MP + 2 MP Triple Rear & 16 MP Front...,Memory Card Not Supported,Android v13
...,...,...,...,...,...,...,...,...,...,...,...,...
1015,1017,Motorola Moto Edge S30 Pro,34990,83.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","8 GB RAM, 128 GB inbuilt",5000 mAh Battery with 68.2W Fast Charging,"6.67 inches, 1080 x 2460 px, 120 Hz Display wi...",64 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,Android v12,No FM Radio
1016,1018,Honor X8 5G,14990,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 480+, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 22.5W Fast Charging,"6.5 inches, 720 x 1600 px Display with Water D...",48 MP + 2 MP + Depth Sensor Triple Rear & 8 MP...,"Memory Card Supported, upto 1 TB",Android v11
1017,1019,POCO X4 GT 5G (8GB RAM + 256GB),28990,85.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Dimensity 8100, Octa Core, 2.85 GHz Processor","8 GB RAM, 256 GB inbuilt",5080 mAh Battery with 67W Fast Charging,"6.6 inches, 1080 x 2460 px, 144 Hz Display wit...",64 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,Memory Card Not Supported,Android v12
1018,1020,Motorola Moto G91 5G,19990,80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.8 inches, 1080 x 2400 px Display with Punch ...",108 MP + 8 MP + 2 MP Triple Rear & 32 MP Front...,"Memory Card Supported, upto 1 TB",Android v12


In [20]:
df1[df1['index'].isin(processor_rows)]# checking for processor promblematic rows(each column's problem rows also reduced)

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
645,647,Nokia 2760 Flip,5490,NaN,"Dual Sim, 3G, 4G, Wi-Fi",1450 mAh Battery,"3.6 inches, 240 x 320 px Display",5 MP Rear & 5 MP Front Camera,"Memory Card Supported, upto 32 GB",Kaios v3.0,Bluetooth,NaN
857,859,LG Folder 2,11999,NaN,"Single Sim, 3G, 4G, Wi-Fi","1 GB RAM, 8 GB inbuilt",1470 mAh Battery,"2.8 inches, 240 x 320 px Display",2 MP Rear Camera,Memory Card Supported,Bluetooth,NaN
882,884,Nokia 5710 XpressAudio,4799,NaN,"Dual Sim, 3G, 4G",No Wifi,Unisoc T107,"48 MB RAM, 128 MB inbuilt",1450 mAh Battery,"2.4 inches, 240 x 320 px Display",0.3 MP Rear Camera,"Memory Card Supported, upto 32 GB"
925,927,Nokia 3310 4G,3999,NaN,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi","256 MB RAM, 512 MB inbuilt",1200 mAh Battery,"2.4 inches, 240 x 320 px Display",2 MP Rear Camera,"Memory Card Supported, upto 32 GB",Bluetooth,Browser


In [21]:
df1.drop([645,857,882,925],inplace=True) # all are feature phone drop them

In [22]:
df1[df1['index'].isin(ram_rows)] # checking for ram promblematic rows

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
439,441,Apple iPhone SE 3 2022,43900,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,"4.7 inches, 750 x 1334 px Display",12 MP Rear & 7 MP Front Camera,Memory Card Not Supported,iOS v15,No FM Radio
483,485,Huawei Mate 50 RS Porsche Design,239999,81.0,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi, NFC, IR Blaster","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor",512 GB inbuilt,4700 mAh Battery with 66W Fast Charging,"6.74 inches, 1212 x 2616 px, 120 Hz Display",50 MP + 48 MP + 13 MP Triple Rear & 13 MP Fron...,"Memory Card (Hybrid), upto 256 GB",Hongmeng OS v3.0
582,584,Nokia 8210 4G,3749,NaN,"Dual Sim, 3G, 4G",No Wifi,Unisoc T107,"48 MB RAM, 128 MB inbuilt",1450 mAh Battery,"2.8 inches, 240 x 320 px Display",0.3 MP Rear Camera,"Memory Card Supported, upto 32 GB"


In [23]:
df1.drop(582,inplace=True) # this was a feature phn so drop it

In [24]:
df1[df1['index'].isin(battery_rows)] # checking for battery promblematic rows , also battery data needs to be shifted by 1

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
111,113,Apple iPhone 12,51999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt","6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14,No FM Radio
149,151,Apple iPhone 12 Mini,40999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt","5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14,No FM Radio
307,309,Apple iPhone 12 (128GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt","6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14,No FM Radio
363,365,Apple iPhone 12 Mini (128GB),45999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt","5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14,No FM Radio
376,378,Nokia 2660 Flip,4649,NaN,"Dual Sim, 3G, 4G",No Wifi,Unisoc T107,"48 MB RAM, 128 MB inbuilt",1450 mAh Battery,"2.8 inches, 240 x 320 px Display",Dual Display,0.3 MP Rear Camera
439,441,Apple iPhone SE 3 2022,43900,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,"4.7 inches, 750 x 1334 px Display",12 MP Rear & 7 MP Front Camera,Memory Card Not Supported,iOS v15,No FM Radio
448,450,Apple iPhone 15 Pro,130990,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",Bionic A16,"8 GB RAM, 128 GB inbuilt","6.06 inches, 1170 x 2532 px, 120 Hz Display wi...",50 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v15,No FM Radio
628,630,Apple iPhone 12 Pro (512GB),139900,80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","6 GB RAM, 512 GB inbuilt","6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v14.0,No FM Radio
754,756,Apple iPod Touch (7th Gen),18900,NaN,Wi-Fi,32 GB inbuilt,"4 inches, 640 x 1136 px Display",8 MP Rear & 1.2 MP Front Camera,iOS v12,No FM Radio,Bluetooth,Browser
762,764,Apple iPhone SE 4,49990,60.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,"6.1 inches, 750 x 1580 px Display",12 MP Rear & 10.8 MP Front Camera,Memory Card Not Supported,iOS v16,No FM Radio


In [25]:
df1.drop([376,754],inplace=True) #drop feature phone and one ipod

In [26]:
temp_df = df1[df1['index'].isin(battery_rows)] # storing problematoc battery rows in temp df

In [27]:
x = temp_df.iloc[:,7:].shift(1,axis=1).values # shifting by 1 from battery and storing value in x

In [28]:
df1.loc[temp_df.index,temp_df.columns[7:]] = x # putting shifting values to df1

In [29]:
df1[df1['index'].isin(battery_rows)] # gotten the shifted data , battery colums became None

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
111,113,Apple iPhone 12,51999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
149,151,Apple iPhone 12 Mini,40999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
307,309,Apple iPhone 12 (128GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
363,365,Apple iPhone 12 Mini (128GB),45999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
439,441,Apple iPhone SE 3 2022,43900,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,None,"4.7 inches, 750 x 1334 px Display",12 MP Rear & 7 MP Front Camera,Memory Card Not Supported,iOS v15
448,450,Apple iPhone 15 Pro,130990,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",Bionic A16,"8 GB RAM, 128 GB inbuilt",None,"6.06 inches, 1170 x 2532 px, 120 Hz Display wi...",50 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v15
628,630,Apple iPhone 12 Pro (512GB),139900,80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","6 GB RAM, 512 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v14.0
762,764,Apple iPhone SE 4,49990,60.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,None,"6.1 inches, 750 x 1580 px Display",12 MP Rear & 10.8 MP Front Camera,Memory Card Not Supported,iOS v16
853,855,Apple iPhone 12 Pro (256GB),119900,80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","6 GB RAM, 256 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v14.0
913,915,Apple iPhone 12 Mini (256GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 256 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14


In [30]:
df1[df1['index'].isin(display_rows)] # checking for battery display rows

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
439,441,Apple iPhone SE 3 2022,43900,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,None,"4.7 inches, 750 x 1334 px Display",12 MP Rear & 7 MP Front Camera,Memory Card Not Supported,iOS v15
448,450,Apple iPhone 15 Pro,130990,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",Bionic A16,"8 GB RAM, 128 GB inbuilt",None,"6.06 inches, 1170 x 2532 px, 120 Hz Display wi...",50 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v15
628,630,Apple iPhone 12 Pro (512GB),139900,80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","6 GB RAM, 512 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP + 12 MP Triple Rear & 12 MP Fron...,Memory Card Not Supported,iOS v14.0
762,764,Apple iPhone SE 4,49990,60.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A15, Hexa Core, 3.22 GHz Processor",64 GB inbuilt,None,"6.1 inches, 750 x 1580 px Display",12 MP Rear & 10.8 MP Front Camera,Memory Card Not Supported,iOS v16
913,915,Apple iPhone 12 Mini (256GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 256 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
914,916,Apple iPhone 12 (256GB),67999,76.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 256 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14


In [31]:
len(display_rows)# set of index of problem rows had 27 index in start , some got removed as we only take price > 3400
# but by shiting battery , all display problem get solved

27

In [32]:
len(camera_rows) # total problem rows in start

64

In [33]:
df1[df1['index'].isin(camera_rows)] # still has 39 problem rows
# 155 271 are feature phn index column , (camera where foldable or dual display written , data is in card column)

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
98,100,Vivo X Fold 5G,106990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","12 GB RAM, 256 GB inbuilt",4600 mAh Battery with 66W Fast Charging,"8.03 inches, 1916 x 2160 px, 120 Hz Display",Foldable Display,50 MP Quad Rear & 16 MP Front Camera,Android v12
111,113,Apple iPhone 12,51999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
149,151,Apple iPhone 12 Mini,40999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
155,157,Nokia 2780 Flip,4990,NaN,"Dual Sim, 3G, 4G, Wi-Fi","Snapdragon QM215, Quad Core, 1.3 GHz Processor","4 GB RAM, 512 MB inbuilt",1450 mAh Battery,"2.7 inches, 240 x 320 px Display",Dual Display,5 MP Rear Camera,"Memory Card Supported, upto 32 GB"
159,161,Oppo Find N2 5G,94990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",4520 mAh Battery with 67W Fast Charging,"7.1 inches, 1792 x 1920 px, 120 Hz Display wit...","Foldable Display, Dual Display",50 MP + 48 MP + 32 MP Triple Rear & 32 MP + 32...,Memory Card Not Supported
236,238,Xiaomi Mix Fold 2 5G,106990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Snapdragon 8+ Gen1 , Octa Core, 3.2 GHz Proce...","12 GB RAM, 256 GB inbuilt",4500 mAh Battery with 67W Fast Charging,"8.02 inches, 1914 x 2160 px, 120 Hz Display wi...","Foldable Display, Dual Display",50 MP + 13 MP + 8 MP Triple Rear & 20 MP Front...,Android v12
271,273,Nokia 2720 V Flip,6199,NaN,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi","Snapdragon 205 , Dual Core, 1.1 GHz Processor","512 MB RAM, 4 GB inbuilt",1500 mAh Battery,"2.8 inches, 240 x 320 px Display",Dual Display,2 MP Rear Camera,Memory Card Supported
306,308,Samsung Galaxy Z Flip 3,69999,84.0,"Single Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 888, Octa Core, 2.84 GHz Processor","8 GB RAM, 128 GB inbuilt",3300 mAh Battery with 15W Fast Charging,"6.7 inches, 1080 x 2640 px, 120 Hz Display wit...","Foldable Display, Dual Display",12 MP + 12 MP Dual Rear & 10 MP Front Camera,Memory Card Not Supported
307,309,Apple iPhone 12 (128GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
321,323,Samsung Galaxy Z Fold 4,154998,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",4400 mAh Battery with 25W Fast Charging,"7.6 inches, 1812 x 2176 px, 120 Hz Display wit...","Foldable Display, Dual Display",50 MP + 12 MP + 10 MP Triple Rear & 10 MP + 4 ...,Android v12


In [34]:
df1.drop([155, 271],inplace=True) # drop the feature phone

In [35]:
temp_df = df1[df1['index'].isin(camera_rows)] # it still has 37 rows , making temp df

In [36]:
temp_df = temp_df[~temp_df['camera'].str.contains('MP')] # all correct data has MP in it so using not for dual or foldable data

In [37]:
df1.loc[temp_df.index, 'camera'] = temp_df['card'].values # changing those valued to the values in card ( not shifted as next columns 'os' has right data)

In [38]:
df1[df1['index'].isin(camera_rows)] # all camera issue is solved

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
98,100,Vivo X Fold 5G,106990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","12 GB RAM, 256 GB inbuilt",4600 mAh Battery with 66W Fast Charging,"8.03 inches, 1916 x 2160 px, 120 Hz Display",50 MP Quad Rear & 16 MP Front Camera,50 MP Quad Rear & 16 MP Front Camera,Android v12
111,113,Apple iPhone 12,51999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
149,151,Apple iPhone 12 Mini,40999,74.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 64 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
159,161,Oppo Find N2 5G,94990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",4520 mAh Battery with 67W Fast Charging,"7.1 inches, 1792 x 1920 px, 120 Hz Display wit...",50 MP + 48 MP + 32 MP Triple Rear & 32 MP + 32...,50 MP + 48 MP + 32 MP Triple Rear & 32 MP + 32...,Memory Card Not Supported
236,238,Xiaomi Mix Fold 2 5G,106990,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Snapdragon 8+ Gen1 , Octa Core, 3.2 GHz Proce...","12 GB RAM, 256 GB inbuilt",4500 mAh Battery with 67W Fast Charging,"8.02 inches, 1914 x 2160 px, 120 Hz Display wi...",50 MP + 13 MP + 8 MP Triple Rear & 20 MP Front...,50 MP + 13 MP + 8 MP Triple Rear & 20 MP Front...,Android v12
306,308,Samsung Galaxy Z Flip 3,69999,84.0,"Single Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 888, Octa Core, 2.84 GHz Processor","8 GB RAM, 128 GB inbuilt",3300 mAh Battery with 15W Fast Charging,"6.7 inches, 1080 x 2640 px, 120 Hz Display wit...",12 MP + 12 MP Dual Rear & 10 MP Front Camera,12 MP + 12 MP Dual Rear & 10 MP Front Camera,Memory Card Not Supported
307,309,Apple iPhone 12 (128GB),55999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt",None,"6.1 inches, 1170 x 2532 px Display with Large ...",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14
321,323,Samsung Galaxy Z Fold 4,154998,NaN,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8+ Gen1, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",4400 mAh Battery with 25W Fast Charging,"7.6 inches, 1812 x 2176 px, 120 Hz Display wit...",50 MP + 12 MP + 10 MP Triple Rear & 10 MP + 4 ...,50 MP + 12 MP + 10 MP Triple Rear & 10 MP + 4 ...,Android v12
322,324,Royole FlexPai 2,109999,87.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 865, Octa Core, 2.84 GHz Processor","8 GB RAM, 128 GB inbuilt",4450 mAh Battery,"7.8 inches, 1440 x 1920 px Display",64 MP + 16 MP + 8 MP Triple Rear & 32 MP Front...,64 MP + 16 MP + 8 MP Triple Rear & 32 MP Front...,"Memory Card Supported, upto 256 GB"
363,365,Apple iPhone 12 Mini (128GB),45999,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Bionic A14, Hexa Core, 3.1 GHz Processor","4 GB RAM, 128 GB inbuilt",None,"5.4 inches, 1080 x 2340 px Display",12 MP + 12 MP Dual Rear & 12 MP Front Camera,Memory Card Not Supported,iOS v14


In [39]:
# all problem rows set we made are done , still has many rows as we have not dropped much
# last 2 columns 'card' and 'os' are left

In [40]:
df1['card'].value_counts() # we can see there are some camera data MP due to shifting ,we gave them to camera above but still present in  card

,count
card,
"Memory Card Supported, upto 1 TB",171
Memory Card Not Supported,123
Android v12,107
"Memory Card Supported, upto 512 GB",105
"Memory Card (Hybrid), upto 1 TB",91
Memory Card Supported,89
"Memory Card Supported, upto 256 GB",87
Android v13,46
Android v11,41


In [41]:
temp_df = df1[df1['card'].str.contains('MP')]

In [42]:
df1.loc[temp_df.index,'card'] = 'Memory Card Not Supported' # changing all MP data to memory card not found

In [43]:
df1['card'].value_counts() #all MP's are gone but still have android, bluetooth(data of os due to shifting)

,count
card,
"Memory Card Supported, upto 1 TB",171
Memory Card Not Supported,149
Android v12,107
"Memory Card Supported, upto 512 GB",105
"Memory Card (Hybrid), upto 1 TB",91
Memory Card Supported,89
"Memory Card Supported, upto 256 GB",87
Android v13,46
Android v11,41


In [44]:
#pd.set_option('display.max_rows', None)

In [45]:
temp_df = df1[~df1['card'].str.contains('Memory Card')] # card column has either data in form of memory card or android (which is of 'os')

In [46]:
df1.loc[temp_df.index,'os'] = temp_df['card'].values # giving android bluetooth values to 'os'

In [47]:
df1.loc[temp_df.index,'card'] = 'Memory Card Not Supported' # making the same data memory card not supported in 'card' column

In [48]:
df1['card'].value_counts() # all issue solved ,all data in form of memory card

,count
card,
Memory Card Not Supported,362
"Memory Card Supported, upto 1 TB",171
"Memory Card Supported, upto 512 GB",105
"Memory Card (Hybrid), upto 1 TB",91
Memory Card Supported,89
"Memory Card Supported, upto 256 GB",87
Memory Card (Hybrid),30
"Memory Card (Hybrid), upto 256 GB",13
"Memory Card (Hybrid), upto 512 GB",11


In [49]:
df1['os'].value_counts() # has memory card in 'os' so make it np.nan , also has bluetooth

,count
os,
Android v12,394
Android v11,274
Android v13,91
Android v10,69
Android v9.0 (Pie),29
Android v10.0,23
iOS v16,15
iOS v15,12
Android v8.1 (Oreo),10


In [50]:
temp_df = df1[df1['os'].str.contains('Memory Card')]

In [51]:
df1.loc[temp_df.index,'os'] = np.nan #making memory card data nan
#df1['os'].value_counts() can check with value_counts each time # now we have all data of os but bluetooth is still there

In [52]:
temp_df = df1[df1['os'] == 'Bluetooth']

In [53]:
df1.loc[temp_df.index,'os'] = np.nan
df1['os'].value_counts() # all bluetooth ,android data is gone (only has os types)

,count
os,
Android v12,394
Android v11,274
Android v13,91
Android v10,69
Android v9.0 (Pie),29
Android v10.0,23
iOS v16,15
iOS v15,12
Android v8.1 (Oreo),10


In [54]:
df1['display'].value_counts() # can check for each columns, all values seem to be fine

,count
display,
"6.67 inches, 1080 x 2400 px, 120 Hz Display with Punch Hole",54
"6.5 inches, 720 x 1600 px Display with Water Drop Notch",36
"6.7 inches, 1080 x 2412 px, 120 Hz Display with Punch Hole",25
"6.52 inches, 720 x 1600 px Display with Water Drop Notch",23
"6.5 inches, 1080 x 2400 px, 90 Hz Display with Punch Hole",22
...,...
"5.86 inches, 720 x 1520 px Display with Large Notch",1
"6.43 inches, 1440 x 3200 px, 120 Hz Display with Punch Hole",1
"6.6 inches, 1080 x 2400 px, 144 Hz Display",1


In [55]:
(982/1020)*100 # uncleaned data rows = 1020 ,cleaned data rows= 982

96.27450980392157

In [56]:
df1

,index,model,price,rating,sim,processor,ram,battery,display,camera,card,os
0,2,OnePlus 11 5G,54999,89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 8 Gen2, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",5000 mAh Battery with 100W Fast Charging,"6.7 inches, 1440 x 3216 px, 120 Hz Display wit...",50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,Memory Card Not Supported,Android v13
1,3,OnePlus Nord CE 2 Lite 5G,19989,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 33W Fast Charging,"6.59 inches, 1080 x 2412 px, 120 Hz Display wi...",64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
2,4,Samsung Galaxy A14 5G,16499,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Exynos 1330, Octa Core, 2.4 GHz Processor","4 GB RAM, 64 GB inbuilt",5000 mAh Battery with 15W Fast Charging,"6.6 inches, 1080 x 2408 px, 90 Hz Display with...",50 MP + 2 MP + 2 MP Triple Rear & 13 MP Front ...,"Memory Card Supported, upto 1 TB",Android v13
3,5,Motorola Moto G62 5G,14999,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.55 inches, 1080 x 2400 px, 120 Hz Display wi...",50 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
4,6,Realme 10 Pro Plus,24999,82.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Dimensity 1080, Octa Core, 2.6 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 67W Fast Charging,"6.7 inches, 1080 x 2412 px, 120 Hz Display wit...",108 MP + 8 MP + 2 MP Triple Rear & 16 MP Front...,Memory Card Not Supported,Android v13
...,...,...,...,...,...,...,...,...,...,...,...,...
1015,1017,Motorola Moto Edge S30 Pro,34990,83.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 8 Gen1, Octa Core, 3 GHz Processor","8 GB RAM, 128 GB inbuilt",5000 mAh Battery with 68.2W Fast Charging,"6.67 inches, 1080 x 2460 px, 120 Hz Display wi...",64 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,Memory Card Not Supported,Android v12
1016,1018,Honor X8 5G,14990,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi","Snapdragon 480+, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with 22.5W Fast Charging,"6.5 inches, 720 x 1600 px Display with Water D...",48 MP + 2 MP + Depth Sensor Triple Rear & 8 MP...,"Memory Card Supported, upto 1 TB",Android v11
1017,1019,POCO X4 GT 5G (8GB RAM + 256GB),28990,85.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC, IR Bl...","Dimensity 8100, Octa Core, 2.85 GHz Processor","8 GB RAM, 256 GB inbuilt",5080 mAh Battery with 67W Fast Charging,"6.6 inches, 1080 x 2460 px, 144 Hz Display wit...",64 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,Memory Card Not Supported,Android v12
1018,1020,Motorola Moto G91 5G,19990,80.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC","Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",5000 mAh Battery with Fast Charging,"6.8 inches, 1080 x 2400 px Display with Punch ...",108 MP + 8 MP + 2 MP Triple Rear & 32 MP Front...,"Memory Card Supported, upto 1 TB",Android v12


In [57]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 982 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   index      982 non-null    int64  
 1   model      982 non-null    object 
 2   price      982 non-null    int64  
 3   rating     879 non-null    float64
 4   sim        982 non-null    object 
 5   processor  982 non-null    object 
 6   ram        982 non-null    object 
 7   battery    971 non-null    object 
 8   display    982 non-null    object 
 9   camera     982 non-null    object 
 10  card       982 non-null    object 
 11  os         967 non-null    object 
dtypes: float64(1), int64(2), object(9)
memory usage: 132.0+ KB


In [58]:
brand_names = df1['model'].str.strip().str.split(' ').str.get(0).str.lower() # model column has brand name as first word , extract it and make it lower so OPPO oppo both same

In [59]:
df1.insert(1,'brand_name',brand_names)  # inserting at 1st index

In [60]:
#df1['brand_name'] = df1['brand_name'].str.lower()  #can do it alag se ir either saath me like above
df1['brand_name'].value_counts() # can see no OPPO oppo issue and we made all brand names

,count
brand_name,
xiaomi,134
samsung,132
vivo,111
realme,97
oppo,88
motorola,52
apple,46
oneplus,42
poco,41


In [61]:
sim_type = df1['sim'].str.strip().str.split(",").str.get(0)
has_5g = df1['sim'].str.contains('5G') # sim column can be use to make 3 new colum
has_nfc = df1['sim'].str.contains('NFC') # all 2 has data in true false
has_ir_blaster = df1['sim'].str.contains('IR Blaster')

In [62]:
df1.insert(6,'sim_type',sim_type)
df1.insert(7,'has_5g',has_5g)
df1.insert(8,'has_nfc',has_nfc)
df1.insert(9,'has_ir_blaster',has_ir_blaster)


In [63]:
processor_name = df1['processor'].str.strip().str.split(',').str.get(0) # procssor column has 3 also be used to make 3 new columns
processor_core = df1['processor'].str.split(',').str.get(1)
processor_speed = df1['processor'].str.split(',').str.get(2)

In [64]:
df1.insert(10,'processor_name',processor_name) # inserting
df1.insert(11,'processor_core',processor_core)
df1.insert(12,'processor_speed',processor_speed)

In [65]:
#df1['processor_name'] = df1['processor_name'].str.strip() # on checking value check we found space in value

In [66]:
temp_df = df1[df1['processor_name'].str.contains('Core')][['processor_name', 'processor_core',	'processor_speed']].shift(1,axis=1)
# sometimes processor columns has 2 or 1 thing ,so making order wrong , so whenever core in processor name we have to do shifting

In [67]:
temp_df.shape

(20, 3)

In [68]:
df1.loc[temp_df.index,['processor_name', 'processor_core',	'processor_speed']] = temp_df.values

In [69]:
df1.loc[856] # at index 856 we found a processor name 28 nm . we google the phone processor name and put it

,856
index,858
brand_name,samsung
model,Samsung Galaxy A01 Core
price,4999
rating,NaN
sim,"Dual Sim, 3G, 4G, VoLTE, Wi-Fi"
sim_type,Dual Sim
has_5g,False
has_nfc,False
has_ir_blaster,False


In [70]:
df1.loc[856,'processor_name'] = 'Mediatek MT6739' # we google the phone processor name and put it

In [71]:
processor_brand = df1['processor_name'].str.split(' ').str.get(0).str.lower() # can make only brand with whole processor name

In [72]:
df1.insert(11,'processor_brand',processor_brand)

In [73]:
df1['processor_core'] = df1['processor_core'].str.strip()

In [74]:
df1['processor_core'] = df1['processor_core'].str.replace('Octa Core Processor','Octa Core').str.replace('Hexa Core Processor','Hexa Core')#on checking value_counts we found these are are same but used with diff names

In [75]:
df1['processor_speed'] = df1['processor_speed'].str.strip().str.split(' ').str.get(0).str.replace('\u2009',' ').str.split(' ').str.get(0).astype(float) # can do by regex too
# processor speed is like 2.2 GHz processor, so we split by space but we still get 2.2 Ghz together due to thin sapce u2009 , so use again and convert to float as 2.2 is float

In [76]:
df1.columns # made these many new columns

Index(['index', 'brand_name', 'model', 'price', 'rating', 'sim', 'sim_type',
       'has_5g', 'has_nfc', 'has_ir_blaster', 'processor_name',
       'processor_brand', 'processor_core', 'processor_speed', 'processor',
       'ram', 'battery', 'display', 'camera', 'card', 'os'],
      dtype='object')

In [77]:
ram_capacity = df1['ram'].str.strip().str.split(',').str.get(0).str.findall(r'\b(\d+)\b').str.get(0)
# ram colum has 6 GB RAM, 128 GB inbuilt . after spliting by comma we can't split by space like above due to unicode thin space . so used regex

In [78]:
df1.insert(16,'ram_capacity',ram_capacity)

In [79]:
df1.head()

,index,brand_name,model,price,rating,sim,sim_type,has_5g,has_nfc,has_ir_blaster,...,processor_core,processor_speed,processor,ram,ram_capacity,battery,display,camera,card,os
0,2,oneplus,OnePlus 11 5G,54999,89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",Dual Sim,True,True,False,...,Octa Core,3.2,"Snapdragon 8 Gen2, Octa Core, 3.2 GHz Processor","12 GB RAM, 256 GB inbuilt",12,5000 mAh Battery with 100W Fast Charging,"6.7 inches, 1440 x 3216 px, 120 Hz Display wit...",50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,Memory Card Not Supported,Android v13
1,3,oneplus,OnePlus Nord CE 2 Lite 5G,19989,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,Octa Core,2.2,"Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",6,5000 mAh Battery with 33W Fast Charging,"6.59 inches, 1080 x 2412 px, 120 Hz Display wi...",64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
2,4,samsung,Samsung Galaxy A14 5G,16499,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,Octa Core,2.4,"Exynos 1330, Octa Core, 2.4 GHz Processor","4 GB RAM, 64 GB inbuilt",4,5000 mAh Battery with 15W Fast Charging,"6.6 inches, 1080 x 2408 px, 90 Hz Display with...",50 MP + 2 MP + 2 MP Triple Rear & 13 MP Front ...,"Memory Card Supported, upto 1 TB",Android v13
3,5,motorola,Motorola Moto G62 5G,14999,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,Octa Core,2.2,"Snapdragon 695, Octa Core, 2.2 GHz Processor","6 GB RAM, 128 GB inbuilt",6,5000 mAh Battery with Fast Charging,"6.55 inches, 1080 x 2400 px, 120 Hz Display wi...",50 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
4,6,realme,Realme 10 Pro Plus,24999,82.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,Octa Core,2.6,"Dimensity 1080, Octa Core, 2.6 GHz Processor","6 GB RAM, 128 GB inbuilt",6,5000 mAh Battery with 67W Fast Charging,"6.7 inches, 1080 x 2412 px, 120 Hz Display wit...",108 MP + 8 MP + 2 MP Triple Rear & 16 MP Front...,Memory Card Not Supported,Android v13


In [80]:
internal_memory = df1['ram'].str.strip().str.split(',').str.get(1).str.strip().str.findall(r'\b(\d+)\b').str.get(0)

In [81]:
df1.insert(17,'internal_memory',internal_memory)

In [82]:
df1['ram_capacity'] = df1['ram_capacity'].astype(float)

In [83]:
df1.drop([486,627],inplace=True)

In [84]:
df1.loc[[483], ['ram_capacity','internal_memory']] = [12.0,'512'] # checked from google

In [85]:
df1['ram_capacity'].value_counts()

,count
ram_capacity,
8.0,339
6.0,234
4.0,215
12.0,86
3.0,54
2.0,32
16.0,9
1.0,7
18.0,2


In [86]:
df1['internal_memory'] = df1['internal_memory'].astype(float)

In [87]:
temp_df = df1[df1['internal_memory'] == 1] # these are TB not GB

In [88]:
df1.loc[temp_df.index,'internal_memory'] = 1024

In [89]:
df1['internal_memory'].value_counts()

,count
internal_memory,
128.0,523
64.0,191
256.0,157
32.0,67
512.0,22
16.0,12
1024.0,5
8.0,1


In [90]:
battery_capacity = df1['battery'].str.strip().str.split('with').str.get(0).str.strip().str.findall(r'\b(\d+)\b').str.get(0).astype(float)
#fast_charging = df1['battery'].str.strip().str.split('with').str.get(1).str.strip().str.findall(r'\d{2,3}')
fast_charging = pd.to_numeric(df['battery'].str.split('with').str.get(1).str.replace('W','', regex=False).str.strip().str.split().str.get(0), errors='coerce')

In [91]:
df1.insert(19,'battery_capacity',battery_capacity) # batter column has 2 things
df1.insert(20,'fast_charging',fast_charging)

In [92]:
"""def fast_charging_extractor(item):

  if type(item) == list:
    if len(item) == 1:
      return item[0]
    else:
      return 0
  else:
    return -1"""

'def fast_charging_extractor(item):\n\n  if type(item) == list:\n    if len(item) == 1:\n      return item[0]\n    else:\n      return 0\n  else:\n    return -1'

In [93]:
#df1['fast_charging'] = df1['fast_charging'].apply(fast_charging_extractor).astype(int)

In [94]:
df1.columns

Index(['index', 'brand_name', 'model', 'price', 'rating', 'sim', 'sim_type',
       'has_5g', 'has_nfc', 'has_ir_blaster', 'processor_name',
       'processor_brand', 'processor_core', 'processor_speed', 'processor',
       'ram', 'ram_capacity', 'internal_memory', 'battery', 'battery_capacity',
       'fast_charging', 'display', 'camera', 'card', 'os'],
      dtype='object')

In [95]:
display_size_inches = df1['display'].str.strip().str.split(',').str.get(0).str.strip().str.split(' ').str.get(0).astype(float)
display_resolution = df1['display'].str.strip().str.split(',').str.get(1).str.strip().str.split('px').str.get(0)
refresh_rate = df1['display'].str.strip().str.split(',').str.get(2).str.strip().str.findall(r'\d{2,3}').str.get(0).apply(lambda x: 60 if pd.isna(x) else x).astype(int)

In [96]:
df1.insert(22,'display_size_inches',display_size_inches) # display column has 3 things
df1.insert(23,'display_resolution',display_resolution)
df1.insert(24,'refresh_rate',refresh_rate)

In [97]:
df1.head()

,index,brand_name,model,price,rating,sim,sim_type,has_5g,has_nfc,has_ir_blaster,...,battery,battery_capacity,fast_charging,display,display_size_inches,display_resolution,refresh_rate,camera,card,os
0,2,oneplus,OnePlus 11 5G,54999,89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",Dual Sim,True,True,False,...,5000 mAh Battery with 100W Fast Charging,5000.0,100.0,"6.7 inches, 1440 x 3216 px, 120 Hz Display wit...",6.70,1440 x 3216,120,50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,Memory Card Not Supported,Android v13
1,3,oneplus,OnePlus Nord CE 2 Lite 5G,19989,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,5000 mAh Battery with 33W Fast Charging,5000.0,33.0,"6.59 inches, 1080 x 2412 px, 120 Hz Display wi...",6.59,1080 x 2412,120,64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
2,4,samsung,Samsung Galaxy A14 5G,16499,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,5000 mAh Battery with 15W Fast Charging,5000.0,15.0,"6.6 inches, 1080 x 2408 px, 90 Hz Display with...",6.60,1080 x 2408,90,50 MP + 2 MP + 2 MP Triple Rear & 13 MP Front ...,"Memory Card Supported, upto 1 TB",Android v13
3,5,motorola,Motorola Moto G62 5G,14999,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,5000 mAh Battery with Fast Charging,5000.0,NaN,"6.55 inches, 1080 x 2400 px, 120 Hz Display wi...",6.55,1080 x 2400,120,50 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
4,6,realme,Realme 10 Pro Plus,24999,82.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,5000 mAh Battery with 67W Fast Charging,5000.0,67.0,"6.7 inches, 1080 x 2412 px, 120 Hz Display wit...",6.70,1080 x 2412,120,108 MP + 8 MP + 2 MP Triple Rear & 16 MP Front...,Memory Card Not Supported,Android v13


In [98]:
def extract_camera(text): # function for getting rear cam count
  if 'Quad' in text:
    return 4
  elif 'Triple' in text:
    return 3
  elif 'Dual' in text:
    return 2
  else:
    return 1

In [99]:
num_rear_cameras = df1['camera'].str.strip().str.split('&').str.get(0).apply(extract_camera)

In [100]:
df1.insert(25,'num_rear_cameras',num_rear_cameras)

In [101]:
def front_camera_count(text): # function for getting front cam count
    text_lower = text.lower()
    if 'dual' in text_lower and 'front camera' in text_lower:
        return 2
    elif 'triple' in text_lower and 'front camera' in text_lower:
        return 3
    elif 'quad' in text_lower and 'front camera' in text_lower:
        return 4
    elif 'front camera' in text_lower:
        return 1  # default for single front camera
    else:
        return 0


In [102]:
num_front_cameras = df1['camera'].str.strip().str.split('&').str.get(1).str.strip().fillna('Missing').apply(front_camera_count)

In [103]:
df1.insert(26,'num_front_cameras',num_front_cameras)

In [104]:
df1.head()

,index,brand_name,model,price,rating,sim,sim_type,has_5g,has_nfc,has_ir_blaster,...,fast_charging,display,display_size_inches,display_resolution,refresh_rate,num_rear_cameras,num_front_cameras,camera,card,os
0,2,oneplus,OnePlus 11 5G,54999,89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",Dual Sim,True,True,False,...,100.0,"6.7 inches, 1440 x 3216 px, 120 Hz Display wit...",6.70,1440 x 3216,120,3,1,50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,Memory Card Not Supported,Android v13
1,3,oneplus,OnePlus Nord CE 2 Lite 5G,19989,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,33.0,"6.59 inches, 1080 x 2412 px, 120 Hz Display wi...",6.59,1080 x 2412,120,3,1,64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
2,4,samsung,Samsung Galaxy A14 5G,16499,75.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,15.0,"6.6 inches, 1080 x 2408 px, 90 Hz Display with...",6.60,1080 x 2408,90,3,1,50 MP + 2 MP + 2 MP Triple Rear & 13 MP Front ...,"Memory Card Supported, upto 1 TB",Android v13
3,5,motorola,Motorola Moto G62 5G,14999,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,NaN,"6.55 inches, 1080 x 2400 px, 120 Hz Display wi...",6.55,1080 x 2400,120,3,1,50 MP + 8 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12
4,6,realme,Realme 10 Pro Plus,24999,82.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,67.0,"6.7 inches, 1080 x 2412 px, 120 Hz Display wit...",6.70,1080 x 2412,120,3,1,108 MP + 8 MP + 2 MP Triple Rear & 16 MP Front...,Memory Card Not Supported,Android v13


In [105]:
#df1[df1['camera'] == 'Foldable Display, Dual Display']
df1.loc[69,'camera'] == '50 MP'

False

In [106]:
temp_df = df1[df1['camera'] == 'Foldable Display, Dual Display']

In [107]:
df1.loc[temp_df.index, 'camera'] = '50 MP'

In [108]:
max_rear_camera_MP = df1['camera'].str.split(' ').str.get(0).str.replace('\u2009',' ').str.split(' ').str.get(0)
max_front_camera_MP = df1['camera'].str.split('&').str.get(1).str.strip().str.split(' ').str.get(0).str.replace('\u2009',' ').str.split(' ').str.get(0)

In [109]:
df1.insert(27,'max_rear_camera_MP',max_rear_camera_MP)
df1.insert(28,'max_front_camera_MP',max_front_camera_MP)

In [110]:
max_front_camera_MP[max_front_camera_MP=='Main'] # have index 613 wrong

,camera
613,Main


In [111]:
df1['max_front_camera_MP']=df1['max_front_camera_MP'].str.replace('Main', '16') # changing it from google

In [112]:
df1.head(2)

,index,brand_name,model,price,rating,sim,sim_type,has_5g,has_nfc,has_ir_blaster,...,display_size_inches,display_resolution,refresh_rate,num_rear_cameras,num_front_cameras,max_rear_camera_MP,max_front_camera_MP,camera,card,os
0,2,oneplus,OnePlus 11 5G,54999,89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",Dual Sim,True,True,False,...,6.70,1440 x 3216,120,3,1,50,16,50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,Memory Card Not Supported,Android v13
1,3,oneplus,OnePlus Nord CE 2 Lite 5G,19989,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,6.59,1080 x 2412,120,3,1,64,16,64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,"Memory Card (Hybrid), upto 1 TB",Android v12


In [113]:
def is_memory_card_supported(text): # checking for card column and making a new column card supported
  if "Not" in text:
    return 0
  else:
    return 1

In [114]:
card_supported = df1['card'].str.strip().str.split(',').str.get(0).apply(is_memory_card_supported)
df1.insert(30,'card_supported',card_supported)

In [115]:
df1.head(2) # added a new column card_supported

,index,brand_name,model,price,rating,sim,sim_type,has_5g,has_nfc,has_ir_blaster,...,display_resolution,refresh_rate,num_rear_cameras,num_front_cameras,max_rear_camera_MP,max_front_camera_MP,camera,card_supported,card,os
0,2,oneplus,OnePlus 11 5G,54999,89.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi, NFC",Dual Sim,True,True,False,...,1440 x 3216,120,3,1,50,16,50 MP + 48 MP + 32 MP Triple Rear & 16 MP Fron...,0,Memory Card Not Supported,Android v13
1,3,oneplus,OnePlus Nord CE 2 Lite 5G,19989,81.0,"Dual Sim, 3G, 4G, 5G, VoLTE, Wi-Fi",Dual Sim,True,False,False,...,1080 x 2412,120,3,1,64,16,64 MP + 2 MP + 2 MP Triple Rear & 16 MP Front ...,1,"Memory Card (Hybrid), upto 1 TB",Android v12


In [116]:
df1['os'].value_counts() # can check for each and replace them with same version

,count
os,
Android v12,394
Android v11,274
Android v13,91
Android v10,69
Android v9.0 (Pie),29
Android v10.0,23
iOS v16,15
iOS v15,12
Android v8.1 (Oreo),10


In [117]:
df1['os'] = df1['os'].str.replace("Android v10.0","Android v10")
df1['os'] = df1['os'].str.replace("Android v11.0","Android v11")
df1['os'] = df1['os'].str.replace("iOS v15.0","iOS v15")
df1['os'] = df1['os'].str.replace("iOS v13.0","iOS v13")
df1['os'] = df1['os'].str.replace("iOS v14.0","iOS v14")
df1['os'] = df1['os'].str.replace("Android v9.0 (Pie)","Android v9 (Pie)")
df1['os'] = df1['os'].replace(['Harmony v2.0', 'HarmonyOS v2.0'], 'HarmonyOS v2', regex=True)

In [119]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 980 entries, 0 to 1019
Data columns (total 33 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   index                980 non-null    int64  
 1   brand_name           980 non-null    object 
 2   model                980 non-null    object 
 3   price                980 non-null    int64  
 4   rating               879 non-null    float64
 5   sim                  980 non-null    object 
 6   sim_type             980 non-null    object 
 7   has_5g               980 non-null    bool   
 8   has_nfc              980 non-null    bool   
 9   has_ir_blaster       980 non-null    bool   
 10  processor_name       960 non-null    object 
 11  processor_brand      960 non-null    object 
 12  processor_core       974 non-null    object 
 13  processor_speed      938 non-null    float64
 14  processor            980 non-null    object 
 15  ram                  980 non-null    object 

In [120]:
df1['max_rear_camera_MP']= df1['max_rear_camera_MP'].astype(float) # chnaging from object to float
df1['max_front_camera_MP']= df1['max_front_camera_MP'].astype(float)

In [121]:
df1['processor_core'].value_counts() # we need ti convert it like octa-8

,count
processor_core,
Octa Core,899
Hexa Core,39
Quad Core,36


In [122]:
def coretype(x):
    if isinstance(x, str):
        if "Octa" in x:
            return 8
        elif "Hexa" in x:
            return 6
        elif "Quad" in x:
            return 4
    return np.nan

In [123]:
df1['processor_core'] = df1['processor_core'].apply(coretype)

In [ ]:
# Drop original columns that have been replaced or are no longer needed

In [124]:
export_df = df1.drop(columns=['index','sim','processor','ram','battery','display','camera','card'])
export_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 980 entries, 0 to 1019
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   brand_name           980 non-null    object 
 1   model                980 non-null    object 
 2   price                980 non-null    int64  
 3   rating               879 non-null    float64
 4   sim_type             980 non-null    object 
 5   has_5g               980 non-null    bool   
 6   has_nfc              980 non-null    bool   
 7   has_ir_blaster       980 non-null    bool   
 8   processor_name       960 non-null    object 
 9   processor_brand      960 non-null    object 
 10  processor_core       974 non-null    float64
 11  processor_speed      938 non-null    float64
 12  ram_capacity         980 non-null    float64
 13  internal_memory      978 non-null    float64
 14  battery_capacity     969 non-null    float64
 15  fast_charging        769 non-null    float64

##  Cleaning Complete: Export Cleaned CVS
Raw, messy scrape into a structured dataset.
* **Dataset Growth:** Started with 11 "composite" columns and expanded them into **25 clean features**.
* **Final Export:** The cleaned data is saved as `smartphone_cleaned_version_for_eda.csv`.

**The data is now ready for the next phase: Exploratory Data Analysis (EDA).**

In [125]:
export_df.to_csv('smartphone_cleaned_version_for_eda.csv',index=False)